# Imports

In [1]:
import os
import re
import json
import time
import math
import threading
import subprocess
from dataclasses import asdict, dataclass
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Config

In [2]:
# --- MELD CSV paths (you run notebooks from /Code, so relative paths look like this) ---
TRAIN_CSV = "../Data/MELD.Raw/train/train_sent_emo.csv"
DEV_CSV   = "../Data/MELD.Raw/dev_sent_emo.csv"
TEST_CSV  = "../Data/MELD.Raw/test_sent_emo.csv"

# --- Columns in MELD CSV (change if yours differs) ---
TEXT_COL  = "Utterance"
LABEL_COL = "Emotion"

# Optional: MELD typically uses "joy"; you want "happiness"
LABEL_REMAP = {"joy": "happiness"}

# --- Experiments output (versioned) ---
EXP_ROOT = "../experiments/text_unimodal"
VERSION  = "v1"  # bump this when you want a new clean run

# --- Training hyperparams ---
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP_NORM = 1.0
SEED = 42

# --- Models to run ---
MODELS_TO_RUN = [
    "bert-base-uncased",
    "distilbert-base-uncased"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cuda


In [3]:
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def slugify(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")

def safe_mkdir_run_dir(root: str) -> str:
    """
    If the dir exists and isn't empty, create root__run2, root__run3, ...
    """
    if not os.path.exists(root):
        os.makedirs(root, exist_ok=True)
        return root

    # exists
    if len(os.listdir(root)) == 0:
        return root

    i = 2
    while True:
        cand = f"{root}__run{i}"
        if not os.path.exists(cand):
            os.makedirs(cand, exist_ok=True)
            return cand
        i += 1

def write_json(path: str, obj: Dict[str, Any]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

set_seed(SEED)


# Load Data + label encoding

In [4]:
def load_split(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"CSV {csv_path} must contain columns: {TEXT_COL}, {LABEL_COL}. Found: {list(df.columns)}")

    df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().str.strip()

    # Optional relabel (e.g., joy -> happiness)
    df[LABEL_COL] = df[LABEL_COL].replace(LABEL_REMAP)

    df.rename(columns={TEXT_COL: "text", LABEL_COL: "emotion"}, inplace=True)
    return df

train_df = load_split(TRAIN_CSV)
dev_df   = load_split(DEV_CSV)
test_df  = load_split(TEST_CSV)

# Label map based on TRAIN only (avoid leaking labels from dev/test)
labels_sorted = sorted(train_df["emotion"].unique().tolist())
label2id = {lab: i for i, lab in enumerate(labels_sorted)}
id2label = {i: lab for lab, i in label2id.items()}

# Encode
train_df["label"] = train_df["emotion"].map(label2id)
dev_df["label"]   = dev_df["emotion"].map(label2id)
test_df["label"]  = test_df["emotion"].map(label2id)

# Drop any rows in dev/test with unseen labels (shouldn't happen, but be safe)
dev_df  = dev_df.dropna(subset=["label"]).copy()
test_df = test_df.dropna(subset=["label"]).copy()
train_df["label"] = train_df["label"].astype(int)
dev_df["label"]   = dev_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

print("Labels:", labels_sorted)
print("Train/Dev/Test:", len(train_df), len(dev_df), len(test_df))


Labels: ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']
Train/Dev/Test: 9989 1109 2610


# Dataset + Dataloaders

In [5]:
class TextEmotionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            return_tensors=None
        )
        enc["labels"] = self.labels[idx]
        return enc

def make_loaders(tokenizer) -> Tuple[DataLoader, DataLoader, DataLoader]:
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")
    train_ds = TextEmotionDataset(train_df, tokenizer)
    dev_ds   = TextEmotionDataset(dev_df, tokenizer)
    test_ds  = TextEmotionDataset(test_df, tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
    dev_loader   = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
    return train_loader, dev_loader, test_loader


# Power Monitor

In [6]:
class NvidiaPowerMonitor:
    """
    Samples GPU power draw via nvidia-smi.
    Logs CSV with columns: t_sec, power_w.
    """
    def __init__(self, interval_s: float = 1.0):
        self.interval_s = interval_s
        self._stop = threading.Event()
        self.samples: List[Tuple[float, float]] = []
        self._t0 = None
        self._thread = None

    @staticmethod
    def _read_power_w() -> float:
        try:
            out = subprocess.check_output(
                "nvidia-smi --query-gpu=power.draw --format=csv,noheader,nounits",
                shell=True
            )
            return float(out.decode("utf-8").strip())
        except Exception:
            return float("nan")

    def start(self):
        self._stop.clear()
        self.samples = []
        self._t0 = time.time()

        def _run():
            while not self._stop.is_set():
                p = self._read_power_w()
                t = time.time() - self._t0
                self.samples.append((t, p))
                time.sleep(self.interval_s)

        self._thread = threading.Thread(target=_run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)

    def stats(self) -> Dict[str, Any]:
        # Filter NaNs (if nvidia-smi unavailable)
        vals = [p for _, p in self.samples if not (isinstance(p, float) and math.isnan(p))]
        if len(vals) == 0:
            return {"available": False}

        avg_power_w = float(np.mean(vals))
        # Energy (kWh) = sum(W)*dt / 3,600,000
        energy_kwh = float(np.sum(vals) * self.interval_s / 3_600_000.0)
        return {
            "available": True,
            "avg_power_w": avg_power_w,
            "energy_kwh": energy_kwh,
            "num_samples": len(vals),
            "interval_s": self.interval_s
        }

    def save_csv(self, path: str):
        df = pd.DataFrame(self.samples, columns=["t_sec", "power_w"])
        df.to_csv(path, index=False)


# Eval + Plots

In [7]:
@torch.no_grad()
def evaluate(model, loader) -> Dict[str, Any]:
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = {k: torch.tensor(v).to(DEVICE) if not torch.is_tensor(v) else v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        losses.append(loss.item())
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = batch["labels"].cpu().numpy()

        all_preds.append(preds)
        all_labels.append(labels)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
        "macro_f1": float(f1_score(all_labels, all_preds, average="macro")),
        "weighted_f1": float(f1_score(all_labels, all_preds, average="weighted")),
        "preds": all_preds,
        "labels": all_labels
    }

def save_confusion_matrix(cm: np.ndarray, labels: List[str], path: str, title: str):
    fig = plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


# Training

Saves best-by-dev-macroF1

In [10]:
def count_params(model) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": int(total), "trainable_params": int(trainable)}

def measure_inference_ms_per_sample(model, loader, num_batches: int = 200) -> float:
    """
    Measures forward pass time only (already-tokenized batches).
    Uses torch.cuda.synchronize() for accurate timing.
    """
    model.eval()
    times = []
    it = iter(loader)

    # Warmup
    for _ in range(10):
        try:
            batch = next(it)
        except StopIteration:
            break
        batch = {k: torch.tensor(v).to(DEVICE) if not torch.is_tensor(v) else v.to(DEVICE) for k, v in batch.items()}
        _ = model(**batch)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for _ in range(num_batches):
        try:
            batch = next(it)
        except StopIteration:
            break
        batch = {k: torch.tensor(v).to(DEVICE) if not torch.is_tensor(v) else v.to(DEVICE) for k, v in batch.items()}

        t0 = time.time()
        _ = model(**batch)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.time()
        times.append((t1 - t0) * 1000.0)

    return float(np.mean(times)) if len(times) else float("nan")

def train_one(model_name: str) -> str:
    model_slug = slugify(model_name)
    run_dir_base = os.path.join(EXP_ROOT, VERSION, model_slug)
    run_dir = safe_mkdir_run_dir(run_dir_base)
    os.makedirs(os.path.join(run_dir, "model"), exist_ok=True)

    # Save config
    cfg = {
        "model_name": model_name,
        "version": VERSION,
        "run_dir": run_dir,
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "seed": SEED,
        "label2id": label2id,
        "id2label": id2label,
        "device": DEVICE,
        "torch_version": torch.__version__,
    }
    write_json(os.path.join(run_dir, "config.json"), cfg)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_loader, dev_loader, test_loader = make_loaders(tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label
    ).to(DEVICE)

    params_info = count_params(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    power_mon = NvidiaPowerMonitor(interval_s=1.0)
    power_mon.start()
    t_train0 = time.time()

    history_rows = []
    best_dev_macro_f1 = -1.0
    best_epoch = -1

    try:
        for epoch in range(1, EPOCHS + 1):
            model.train()
            epoch_loss = 0.0

            for batch in tqdm(train_loader, desc=f"{model_slug} | epoch {epoch}/{EPOCHS}"):
                batch = {k: torch.tensor(v).to(DEVICE) if not torch.is_tensor(v) else v.to(DEVICE) for k, v in batch.items()}

                optimizer.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(**batch)
                    loss = outputs.loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                epoch_loss += loss.item()

            train_loss = epoch_loss / max(1, len(train_loader))
            dev_metrics = evaluate(model, dev_loader)

            row = {
                "epoch": epoch,
                "train_loss": float(train_loss),
                "dev_loss": dev_metrics["loss"],
                "dev_accuracy": dev_metrics["accuracy"],
                "dev_macro_f1": dev_metrics["macro_f1"],
                "dev_weighted_f1": dev_metrics["weighted_f1"],
            }
            history_rows.append(row)
            pd.DataFrame(history_rows).to_csv(os.path.join(run_dir, "history.csv"), index=False)

            # Save best checkpoint by dev macro F1
            if dev_metrics["macro_f1"] > best_dev_macro_f1:
                best_dev_macro_f1 = dev_metrics["macro_f1"]
                best_epoch = epoch
                model.save_pretrained(os.path.join(run_dir, "model"))
                tokenizer.save_pretrained(os.path.join(run_dir, "model"))

    finally:
        t_train1 = time.time()
        power_mon.stop()
        power_mon.save_csv(os.path.join(run_dir, "power_samples.csv"))

    train_time_s = float(t_train1 - t_train0)
    power_stats = power_mon.stats()

    # Load best model for final test eval
    best_model = AutoModelForSequenceClassification.from_pretrained(os.path.join(run_dir, "model")).to(DEVICE)
    test_metrics = evaluate(best_model, test_loader)

    # Save classification report + confusion matrix on test
    rep = classification_report(
        test_metrics["labels"],
        test_metrics["preds"],
        target_names=[id2label[i] for i in range(len(id2label))],
        output_dict=True,
        zero_division=0
    )
    write_json(os.path.join(run_dir, "classification_report_test.json"), rep)

    cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])
    save_confusion_matrix(
        cm,
        labels=[id2label[i] for i in range(len(id2label))],
        path=os.path.join(run_dir, "confusion_matrix_test.png"),
        title=f"Confusion Matrix (Test) — {model_name}"
    )

    # Inference time (batch size 1) on a smaller loader for realism
    bs1_loader = DataLoader(
        TextEmotionDataset(test_df, AutoTokenizer.from_pretrained(os.path.join(run_dir, "model"))),
        batch_size=1,
        shuffle=False,
        collate_fn=DataCollatorWithPadding(tokenizer=AutoTokenizer.from_pretrained(os.path.join(run_dir, "model")))
    )
    infer_ms = measure_inference_ms_per_sample(best_model, bs1_loader, num_batches=200)

    metrics_out = {
        "model_name": model_name,
        "params": params_info,
        "best_epoch": best_epoch,
        "best_dev_macro_f1": best_dev_macro_f1,
        "train_time_s": train_time_s,
        "power": power_stats,
        "test": {
            "loss": test_metrics["loss"],
            "accuracy": test_metrics["accuracy"],
            "macro_f1": test_metrics["macro_f1"],
            "weighted_f1": test_metrics["weighted_f1"],
        },
        "inference_ms_per_sample_bs1": infer_ms,
    }
    write_json(os.path.join(run_dir, "metrics.json"), metrics_out)

    print(f"[DONE] {model_name} -> {run_dir}")
    print("Test macro F1:", metrics_out["test"]["macro_f1"])
    print("Power available:", power_stats.get("available", False))
    return run_dir


# Run BERT + DistilBERT

In [11]:
run_dirs = []
for m in MODELS_TO_RUN:
    run_dirs.append(train_one(m))

run_dirs

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 798.88it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

[DONE] bert-base-uncased -> ../experiments/text_unimodal\v1\bert_base_uncased
Test macro F1: 0.42891996973528
Power available: True


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 848.06it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Andrei\AppData\Local\Temp\ipykernel_2196\1705181319.py:91: FutureWarning: `torch.

[DONE] distilbert-base-uncased -> ../experiments/text_unimodal\v1\distilbert_base_uncased
Test macro F1: 0.42085327501714076
Power available: True


['../experiments/text_unimodal\\v1\\bert_base_uncased',
 '../experiments/text_unimodal\\v1\\distilbert_base_uncased']